# Amaliy mashgʻulot 2. Matnni tozalash, normallashtirish va BPE

2-laboratoriyada tozalash qadamlarini yoqib, BPE harflarni soʻz qismlariga qanday birlashtirishini koʻrdingiz. Bu yerda xuddi shuni Python kodida bajaramiz:

1. bir xil koʻrinadigan satrlar nima uchun har xil boʻlishini koʻramiz (Unicode, NFC, apostrof variantlari) va kirill yozuvini lotinga oʻgiramiz;
2. kichik funksiyalardan normallashtirish konveyerini yigʻamiz va tiplar soni qanday kamayishini sanaymiz;
3. BPE ni noldan yozamiz va har bir birlashtirishni chop etamiz;
4. birlashtirishlarni yangi soʻzlarga qoʻllaymiz va `tokenizers` kutubxonasi bilan solishtiramiz;
5. kattaroq korpusda fertillikni (bir soʻzga nechta token toʻgʻri kelishini) hisoblaymiz.

*«NLP algoritmlari» kursining 2-laboratoriyasiga amaliy mashgʻulot. Kataklarni yuqoridan pastga ishga tushiring: Shift+Enter yoki «Runtime → Run all». Javoblaringizni saqlash uchun «File → Save a copy in Drive» ni bosing.*

## 1. Unicode: koʻrinishi bir xil, ichi har xil

Kompyuter harflarni emas, Unicode belgilari kodlarini saqlaydi. Oʻzbek lotin yozuvida `oʻ` va `gʻ` dagi belgi rasman U+02BB (`ʻ`), tutuq belgisi esa U+02BC (`ʼ`). Amalda esa matnlarda oddiy apostrof `'`, teskari tirnoq `` ` `` va tirnoqlar `‘ ’` uchraydi. Ular **harf emas, tinish belgisi** hisoblanadi: `\w+` muntazam ifodasi `o'zbek` ni ikkiga boʻladi, tinish belgilarini olib tashlash esa `oʻ` ni `o` ga aylantiradi. Quyida apostrofning olti xil varianti.

In [ ]:
import re
import unicodedata

variants = ["o'zbek", "o`zbek", "o‘zbek", "o’zbek", "oʼzbek", "oʻzbek"]
for v in variants:
    c = v[1]
    parts = re.findall(r"\w+", v)     # \w+ bilan soʻzlarga ajratish
    print(f"{v:<8} U+{ord(c):04X} {unicodedata.name(c):<28} toifa={unicodedata.category(c)}  →  {parts}")

Qoida: `o`, `g` dan keyin kelgan har qanday apostrof `ʻ` ga, boshqa harflar orasidagisi esa tutuq belgisi `ʼ` ga aylanadi (`san'at` → `sanʼat`). Kirill yozuvidagi matnni lotinga oʻgirish uchun harflar jadvali va ikkita kontekst qoidasi kerak: `е` soʻz boshida va unlidan keyin `ye` boʻladi (`ер` → `yer`, `поезд` → `poyezd`), `ц` unlidan keyin `ts`, boshqa holatda `s`. Oʻzlashma soʻzlarda istisnolar koʻp, shuning uchun bu yerda faqat muntazam qoidalar bor. Oxirida NFC: kirill `ў` baʼzan ikki belgi bilan (`у` + U+0306) yoziladi.

In [ ]:
APOS = "'`‘’ʼʻ´"

def unify_apostrophes(text):
    # o, g dan keyin: oʻ, gʻ (U+02BB)
    text = re.sub(r"([oOgG])[" + APOS + "]", r"\1ʻ", text)
    # boshqa harflar orasida: tutuq belgisi ʼ (U+02BC)
    text = re.sub(r"(?<=[a-fh-np-zA-FH-NP-Z])[" + APOS + r"](?=[a-zA-Z])", "ʼ", text)
    return text

CYR2LAT = {"а": "a", "б": "b", "в": "v", "г": "g", "д": "d", "ё": "yo", "ж": "j", "з": "z", "и": "i",
           "й": "y", "к": "k", "л": "l", "м": "m", "н": "n", "о": "o", "п": "p", "р": "r", "с": "s",
           "т": "t", "у": "u", "ф": "f", "х": "x", "ч": "ch", "ш": "sh", "ъ": "ʼ", "ь": "", "э": "e",
           "ю": "yu", "я": "ya", "ў": "oʻ", "қ": "q", "ғ": "gʻ", "ҳ": "h"}
CYR_VOWELS = "аеёиоуэюяў"

def translit_word(word):
    low, out = word.lower(), []
    for i, ch in enumerate(low):
        prev = low[i - 1] if i > 0 else ""
        if ch == "е":                      # soʻz boshida va unlidan keyin: ye
            lat = "ye" if (i == 0 or prev in CYR_VOWELS + "ъь") else "e"
        elif ch == "ц":                    # unlidan keyin ts, aks holda s
            lat = "ts" if prev in CYR_VOWELS else "s"
        else:
            lat = CYR2LAT.get(ch, ch)
        if word[i].isupper() and lat:
            lat = lat[0].upper() + lat[1:]
        out.append(lat)
    res = "".join(out)
    return res.upper() if len(word) > 1 and word.isupper() else res

def cyr_to_lat(text):
    return re.sub(r"[А-Яа-яЁёЎўҚқҒғҲҳ]+", lambda m: translit_word(m.group()), text)

def to_nfc(text):
    return unicodedata.normalize("NFC", text)

tests = {"Ўзбекистон": "Oʻzbekiston", "ЎЗБЕКИСТОН": "OʻZBEKISTON", "қишлоқ": "qishloq", "ғалаба": "gʻalaba",
         "ҳаёт": "hayot", "шеър": "sheʼr", "маъно": "maʼno", "поезд": "poyezd", "ер": "yer",
         "Ёшлар": "Yoshlar", "Тошкент": "Toshkent", "концерт": "konsert", "милиция": "militsiya"}
for cyr, lat in tests.items():
    got = cyr_to_lat(cyr)
    print(f"{cyr:<11} → {got:<12} {'✓' if got == lat else '✗ kutilgan: ' + lat}")

print()
for s in ["O'zbekiston", "go`zal", "bog‘", "San'at", "maʻno"]:
    print(f"{s:<12} → {unify_apostrophes(s)}")
dec = "у\u0306збек"
print("\nNFC:", dec, len(dec), "→", to_nfc(dec), len(to_nfc(dec)), "|", cyr_to_lat(to_nfc(dec)))

## 2. Normallashtirish konveyeri

Laboratoriyadagi «iflos» matnni olamiz. Har bir tozalash qadami «matn → matn» funksiyasi, konveyer esa ularning roʻyxati. Tokenlarni laboratoriyadagidek oddiy probel boʻyicha ajratamiz, shuning uchun ketma-ket ikki probel boʻsh token `''` beradi. Har bir qadamdan keyin **tokenlar** (barcha soʻzlar ketma-ket) va **tiplar** (turli soʻzlar) sonini sanaymiz. Baʼzi qadamlar tiplar sonini darhol kamaytirmaydi: «Ўзбекистон:» lotinga oʻgirilgach ham ikki nuqta bilan qoladi, foydasi tinish belgilari olib tashlangandan keyin koʻrinadi.

In [ ]:
SAMPLE = "O'zbekiston — go'zal yurt!  Oʻzbekiston, O`zbekiston va Ўзбекистон: bir so‘z, toʻrt xil yozuv.   Toshkentda 25 ta katta bog' bor. BOGʻDA gullar ochildi, bogʻda — bolalar. Batafsil: https://kun.uz/news/2024 . San'at va sanʼat ham bir soʻz."
STOP = {"va", "ham", "bu", "bir", "ta", "bilan", "uchun", "esa", "u", "edi", "bor"}

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
STEPS = [
    ("kichik harflar",          lambda t: t.lower()),
    ("kirill → lotin",          cyr_to_lat),
    ("apostroflar",             unify_apostrophes),
    ("havolalarni olib tashlash", lambda t: re.sub(r"https?://\S+", "", t)),
    ("tinish belgilari",        lambda t: re.sub(r"[^\w\s]", "", t)),
    ("raqamlar",                lambda t: re.sub(r"\d+", "", t)),
    ("ortiqcha probellar",      lambda t: re.sub(r" +", " ", t).strip()),
    ("toʻxtash soʻzlari",       lambda t: " ".join(w for w in t.split(" ") if w not in STOP)),
]

def run_steps(text, names):
    fns = dict(STEPS)
    for name in names:
        text = fns[name](text)
    return text

def stats(text):
    toks = text.split(" ")
    return len(toks), len(set(toks))

text = SAMPLE
rows = [("dastlabki matn",) + stats(text)]
for name, fn in STEPS:
    text = fn(text)
    rows.append((name,) + stats(text))
table = pd.DataFrame(rows, columns=["bosqich", "tokenlar", "tiplar"])
print(table.to_string(index=False))
print()
print("Barcha qadamlardan keyingi tokenlar:")
print(text.split(" "))
print()
# Tartib muhim: tinish belgilari apostroflardan oldin olib tashlansa, oʻ va gʻ harflari yoʻqoladi
good = run_steps(SAMPLE, ["kichik harflar", "kirill → lotin", "apostroflar", "havolalarni olib tashlash", "tinish belgilari"])
bad = run_steps(SAMPLE, ["kichik harflar", "kirill → lotin", "havolalarni olib tashlash", "tinish belgilari", "apostroflar"])
print("Tinish belgilari apostroflardan oldin, buzilgan tiplar:", sorted(set(bad.split()) - set(good.split())))

plt.figure(figsize=(8, 3))
plt.bar(range(len(table)), table["tiplar"], color="#2B55C9")
plt.xticks(range(len(table)), table["bosqich"], rotation=35, ha="right")
plt.title("Har bir qadamdan keyingi tiplar soni")
plt.tight_layout()
plt.show()

## 3. BPE noldan: oʻqitish

BPE (Byte Pair Encoding) alohida harflar va soʻz oxiri belgisi `</w>` dan iborat lugʻatdan boshlaydi. Har bir qadamda u qoʻshni belgilarning barcha juftliklarini (soʻz chastotalarini hisobga olib) sanaydi va eng koʻp uchraydiganini birlashtiradi:

$$(a, b)^* = \arg\max_{(a,b)} \sum_{w} f(w)\cdot \mathrm{count}_w(a, b), \qquad |V| = |\text{alifbo}| + k.$$

Korpus laboratoriyadagi bilan bir xil: soʻz va uning chastotasi. Chastotalar teng boʻlsa, oldinroq uchragan juftlik tanlanadi (laboratoriyada ham shunday).

In [ ]:
corpus = [("kitob", 6), ("kitoblar", 4), ("kitoblarda", 3), ("kitobingiz", 2), ("kitobimiz", 2), ("kitobda", 3),
          ("kitobdan", 2), ("maktab", 5), ("maktablar", 3), ("maktabingiz", 2), ("maktabda", 4), ("maktabdan", 3),
          ("maktabimiz", 2), ("bolalar", 4), ("bolalarimiz", 2), ("bolada", 2)]
new_words = ["kitoblaringizdan", "maktablarimizdan", "bolalarda", "olmalar", "kitobxon"]

In [ ]:
from collections import Counter

def symbols(word):
    return tuple(word) + ("</w>",)

def pair_counts(vocab):
    counts = Counter()
    for word, freq in vocab.items():
        for a, b in zip(word, word[1:]):
            counts[a, b] += freq
    return counts

def merge_word(word, pair):
    out, i = [], 0
    while i < len(word):
        if i + 1 < len(word) and (word[i], word[i + 1]) == pair:
            out.append(word[i] + word[i + 1])
            i += 2
        else:
            out.append(word[i])
            i += 1
    return tuple(out)

def train_bpe(corpus, num_merges=None, verbose=0):
    vocab = {symbols(w): f for w, f in corpus}
    alphabet = sorted({s for w in vocab for s in w})
    n_words = sum(f for _, f in corpus)
    avg_len = lambda: sum(len(w) * f for w, f in vocab.items()) / n_words
    merges, history = [], [(0, len(alphabet), avg_len())]
    while num_merges is None or len(merges) < num_merges:
        counts = pair_counts(vocab)
        if not counts:
            break
        best = max(counts, key=counts.get)        # chastotalar teng boʻlsa, oldinroq uchragan juftlik tanlanadi
        if counts[best] < 2:
            break
        if len(merges) < verbose:
            top = ", ".join(f"{a}+{b}:{c}" for (a, b), c in counts.most_common(4))
            print(f"qadam {len(merges) + 1:>2}. koʻp juftliklar: {top}")
            print(f"     birlashtirish «{best[0]}» + «{best[1]}» → «{best[0] + best[1]}», lugʻat = {len(alphabet) + len(merges) + 1}")
        vocab = {merge_word(w, best): f for w, f in vocab.items()}
        merges.append(best)
        history.append((len(merges), len(alphabet) + len(merges), avg_len()))
    return merges, alphabet, vocab, history

print("Boshlangʻich alifbo:", " ".join(sorted({s for w, _ in corpus for s in symbols(w)})))
print()
merges, alphabet, seg_vocab, history = train_bpe(corpus, num_merges=8, verbose=8)

Oxirigacha (kamida ikki marta uchraydigan juftliklar qolguncha) oʻqitamiz va laboratoriyadagi grafikdagi egri chiziqlarni chizamiz: lugʻat har bir birlashtirishda 1 taga oʻsadi, bir soʻzga tokenlar soni esa tobora sekin kamayadi.

In [ ]:
merges, alphabet, seg_vocab, history = train_bpe(corpus)
print(f"oʻrganilgan birlashtirishlar: {len(merges)}; |V| = {len(alphabet)} + {len(merges)} = {len(alphabet) + len(merges)}")
print()
print("Barcha birlashtirishlardan keyin korpus soʻzlari:")
for (w, f), seg in zip(corpus, seg_vocab):
    print(f"   {w:<12} ×{f}  →  {' | '.join(seg)}")

k_axis = [h[0] for h in history]
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(k_axis, [h[1] for h in history], color="#2B55C9")
ax[0].set_title("Lugʻat hajmi"); ax[0].set_xlabel("birlashtirishlar"); ax[0].set_ylabel("lugʻat hajmi")
ax[1].plot(k_axis, [h[2] for h in history], color="#17907E")
ax[1].set_title("Bir soʻzga tokenlar"); ax[1].set_xlabel("birlashtirishlar"); ax[1].set_ylabel("bir soʻzga tokenlar (</w> bilan)")
plt.tight_layout()
plt.show()

### Yangi soʻzlarga qoʻllash

Yangi soʻz harflarga ajratiladi va birlashtirishlar **aynan oʻrganilgan tartibda** qoʻllanadi. Faqat ishlagan birlashtirishlarni chop etamiz. Korpusda boʻlmagan harflar alohida belgi boʻlib qoladi: klassik BPE ularni `[UNK]` bilan almashtirardi.

In [ ]:
def bpe_segment(word, merges, verbose=False):
    syms = symbols(word)
    for rank, pair in enumerate(merges, 1):
        if pair in zip(syms, syms[1:]):
            syms = merge_word(syms, pair)
            if verbose:
                print(f"   №{rank:<3} {pair[0]} + {pair[1]:<8} → {' '.join(syms)}")
    return list(syms)

for w in new_words:
    unknown = sorted(set(w) - set(alphabet))
    print(w, f"({len(symbols(w))} belgi)", f"  alifboda yoʻq: {unknown}" if unknown else "")
    seg = bpe_segment(w, merges, verbose=True)
    print("   natija:", " | ".join(seg))
    print()

## 4. Xuddi shu `tokenizers` kutubxonasida

Hugging Face `tokenizers` kutubxonasi BPE ni xuddi shu korpusda oʻqitadi. U Colabda oldindan oʻrnatilgan. Agar u boʻlmasa (internetsiz ishga tushirilganda), katak shunchaki tashlab ketiladi. Birlashtirishlar va boʻlinishlarni biznikilar bilan solishtiring. Kutubxonada `</w>` boshidanoq oxirgi harfga yopishtirilgan, shuning uchun juftliklar biroz boshqacha sanaladi: masalan, `iz` faqat soʻz oxirida (`iz</w>`) oʻrganiladi va «kitoblaringizdan» oʻrtasida `i | z` ga boʻlinadi. Kutubxona lugʻatida boʻlmagan belgilar esa `[UNK]` ga aylanadi (masalan, korpusda uchramagan `x` harfi).

In [ ]:
try:
    from tokenizers import Tokenizer, models, trainers, pre_tokenizers
    HAVE_TOKENIZERS = True
except ImportError:
    HAVE_TOKENIZERS = False
    print("tokenizers kutubxonasi oʻrnatilmagan: katak tashlab ketildi (Colabda u bor).")

if HAVE_TOKENIZERS:
    import json
    tok = Tokenizer(models.BPE(unk_token="[UNK]", end_of_word_suffix="</w>"))
    tok.pre_tokenizer = pre_tokenizers.WhitespaceSplit()
    trainer = trainers.BpeTrainer(vocab_size=1000, min_frequency=2, special_tokens=["[UNK]"],
                                  end_of_word_suffix="</w>", show_progress=False)
    tok.train_from_iterator([w for w, f in corpus for _ in range(f)], trainer=trainer)
    lib_merges = json.loads(tok.to_str())["model"]["merges"]
    print("kutubxonadagi birlashtirishlar:", len(lib_merges), "| |V| =", tok.get_vocab_size())
    print("   ", [" + ".join(m) if isinstance(m, list) else m for m in lib_merges[:8]])
    print()
    for w in new_words:
        print(f"{w:<18} bizning BPE: {' | '.join(bpe_segment(w, merges)):<36} tokenizers: {' | '.join(tok.encode(w).tokens)}")

### Kattaroq korpusda fertillik

**Fertillik** — bir soʻzga toʻgʻri keladigan tokenlarning oʻrtacha soni. Butun soʻzlardan iborat lugʻatda u 1 ga teng, lekin har bir yangi soʻz notanish (OOV). BPE ni 30 ta gapda oʻqitib, boshqa 8 ta gapda tekshiramiz. Yolgʻiz `</w>` ni sanamaymiz: haqiqiy tokenizatorlarda u oxirgi harfga yopishtirilgan. `ʻ` alohida Unicode belgisi, shuning uchun K = 0 da `oʻ` ikki belgi hisoblanadi, BPE esa `o + ʻ` juftligini tezda birlashtiradi.

Oʻz matnlaringizni chapdagi «Files» panelidan yuklashingiz (masalan, `texts.txt`) va `open("texts.txt", encoding="utf-8")` bilan oʻqishingiz mumkin.

In [ ]:
train_text = """
Bolalar maktabga ertalab borishadi. Maktabimizda katta kutubxona bor. Kutubxonada kitoblar juda koʻp.
Oʻquvchilar kitoblarini sumkalariga solishdi. Oʻqituvchimiz bizga yangi kitob berdi. Biz kitoblarimizni ehtiyot qilamiz.
Darsdan keyin doʻstlarim bilan bogʻda oʻynadik. Bogʻimizda olma va oʻrik daraxtlari bor. Bahorda daraxtlar gullaydi.
Onam bozordan olma va uzum olib keldi. Otam har kuni ishga avtobusda boradi.
Toshkent Oʻzbekistonning poytaxti. Samarqand va Buxoro qadimiy shaharlardir. Shaharlarimizda yangi maktablar qurilmoqda.
Talabalar universitetda til va adabiyotni oʻrganishadi. Men kecha kutubxonadan ikkita kitob oldim.
Kitoblarimdan birini ukamga berdim. Ukam kitobni qiziqib oʻqidi. Yozda buvimnikiga qishloqqa boramiz.
Qishlogʻimizda bolalar daryoda suzishadi. Buvim bizga ertak aytib beradi. Ertaklarda yaxshilik doim gʻalaba qiladi.
Daftarlarimizni stolga qoʻydik. Oʻqituvchilarimiz bizga doim yordam berishadi. Sinfimizda yigirma besh oʻquvchi bor.
Har bir oʻquvchining oʻz daftari bor. Kitob oʻqish bilimni oshiradi. Bolalarning kitoblari stol ustida turibdi.
Maktabdan keyin uyga qaytdik. Kechqurun oilamiz bilan choy ichdik.
"""
test_text = """
Kutubxonachilar kitoblarni tartib bilan joylashtirishdi. Maktablarimizdagi kutubxonalar yangilandi.
Doʻstlarimizning bogʻlarida olmalar pishdi. Oʻquvchilarimizga yangi daftarlar berildi. Shaharlardan qishloqlarga avtobuslar qatnaydi.
Ukalarimiz ertaklarni yaxshi koʻrishadi. Universitetlarimizda talabalar koʻpaydi. Kitoblaringizdan birini menga bering.
"""

def words(text):
    return re.findall(r"[a-zʻʼ]+", unify_apostrophes(cyr_to_lat(to_nfc(text))).lower())

In [ ]:
import numpy as np

train_words, test_words = words(train_text), words(test_text)
freq = Counter(train_words)
big_corpus = list(freq.items())
big_merges, big_alphabet, _, _ = train_bpe(big_corpus)
print(f"oʻqitishdagi soʻzlar: {len(train_words)}, tip: {len(freq)}; testdagi soʻzlar: {len(test_words)}")
print(f"oʻrganilgan birlashtirishlar: {len(big_merges)}, |V| = {len(big_alphabet) + len(big_merges)}")
print(f"OOV ulushi (soʻz butunligicha uchramagan): {np.mean([w not in freq for w in test_words]):.0%}")
print(f"notanish harfli test soʻzlari: {[w for w in test_words if set(w) - set(big_alphabet)]}")

def fertility(word_list, merge_list):
    # yolgʻiz </w> sanalmaydi
    return np.mean([sum(s != "</w>" for s in bpe_segment(w, merge_list)) for w in word_list])

Ks = sorted(set([0, 10, 25, 50, 100, 150, 200, 300, len(big_merges)]) & set(range(len(big_merges) + 1)))
fert = [(K, fertility(train_words, big_merges[:K]), fertility(test_words, big_merges[:K])) for K in Ks]
print()
print(pd.DataFrame(fert, columns=["K", "fertillik (oʻqitish)", "fertillik (test)"]).round(2).to_string(index=False))

print()
for w in test_words[:12]:
    print(f"   {w:<20} → {' | '.join(bpe_segment(w, big_merges))}")

plt.figure(figsize=(6, 3.2))
plt.plot(Ks, [f[1] for f in fert], "o-", label="oʻqitish")
plt.plot(Ks, [f[2] for f in fert], "s-", label="test")
plt.axhline(1, color="gray", ls="--", label="butun soʻzlar")
plt.xlabel("birlashtirishlar soni K"); plt.ylabel("fertillik"); plt.title("Bir soʻzga nechta token toʻgʻri keladi")
plt.legend(); plt.tight_layout(); plt.show()

### Haqiqiy modelda

Quyidagi katak faqat Colabda ishlaydi (internet kerak). U model vaznlarisiz ikkita haqiqiy tokenizatorni yuklab oladi: GPT-2 (baytli BPE, 50 257 token, fayl taxminan 1,4 MB) va XLM-RoBERTa (SentencePiece, 250 002 token, taxminan 9 MB). Bir xil iboraning oʻzbekcha, ruscha va inglizcha variantlarida fertillikni solishtiring.

In [ ]:
samples = {
    "uz": "Maktabimiz oʻqituvchilari oʻquvchilarga yangi darsliklar sovgʻa qilishdi.",
    "ru": "Учителя нашей школы подарили ученикам новые учебники.",
    "en": "The teachers of our school gave the students new textbooks.",
}
try:
    from tokenizers import Tokenizer
    for name, model_id in [("GPT-2", "openai-community/gpt2"), ("XLM-RoBERTa", "FacebookAI/xlm-roberta-base")]:
        tk = Tokenizer.from_pretrained(model_id)
        print(name)
        for lang, text in samples.items():
            enc = tk.encode(text, add_special_tokens=False)
            n = len(text.split())
            print(f"   {lang}: {len(enc.tokens):3d} token / {n} soʻz = {len(enc.tokens) / n:.2f}")
        if "xlm" in model_id:
            first = list(samples)[0]
            print("   ", tk.encode(samples[first], add_special_tokens=False).tokens)
except Exception as e:
    print("Tokenizatorni yuklab boʻlmadi (Colabda internet kerak):", e)

## 5. Topshiriqlar

1. `STEPS` ga «sonlar → `<NUM>`» qadamini (olib tashlash emas, almashtirish) tinish belgilari olib tashlangandan keyin qoʻshing, aks holda burchakli qavslar ham oʻchib ketadi. Tokenlar va tiplar soni qanday oʻzgardi? Nima uchun bunday almashtirish olib tashlashdan yaxshiroq boʻlishi mumkin?
2. Kirill yozuvidagi gapni yozing (masalan, `Ўқитувчи шеър ўқиди.`) va uni `cyr_to_lat` bilan oʻgiring. Keyin `ц` va `е` qoidalarisiz (oddiy jadval bilan) oʻgirib, qaysi soʻzlar xato chiqishini toping.
3. Barcha chastotalar 1 ga teng boʻlgan `corpus` da BPE ni oʻqiting. Dastlabki 10 ta birlashtirish va «kitoblaringizdan» soʻzining boʻlinishi qanday oʻzgardi? BPE nima uchun soʻz chastotalarini hisobga olishini tushuntiring.
4. Alifbodan tashqaridagi belgilarni `[UNK]` bilan almashtiradigan `segment_with_unk(word, merges, alphabet)` funksiyasini yozing. Uni 3-boʻlimdagi `new_words` da tekshiring: qaysi soʻzlarda `[UNK]` paydo boʻldi va ular tokenlarining qancha ulushi notanish?
5. 4-boʻlim maʼlumotlari boʻyicha test soʻzlaridagi fertillik K = 0 dagidan bir yarim baravar kichik boʻladigan eng kichik birlashtirishlar soni K ni toping. Bunda lugʻatda nechta belgi bor? Nima uchun test egri chizigʻi doim oʻqitish egri chizigʻidan yuqorida?

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing